In [4]:
import scipy as sp
import numpy as np
import numpy.linalg as la
import sympy as smp
import scipy.stats as stats
import matplotlib.pyplot as plt

In [11]:
# tensor -> matrix

def unfold(M, mode=0):
    m = np.moveaxis(M, mode, 0).reshape(M.shape[mode], -1)
    return m

mat = np.array([
    [
        [1, 5],
        [3, 7]
    ],
    [
        [2, 6],
        [4, 8]
    ]
])

mat, unfold(mat)


(array([[[1, 5],
         [3, 7]],
 
        [[2, 6],
         [4, 8]]]),
 array([[1, 5, 3, 7],
        [2, 6, 4, 8]]))

In [19]:
def factor_matrices(M):
    factors = []

    for mode in range(M.ndim):
        A = unfold(M, mode)
        U, _, _ = la.svd(A)
        factors.append(U)
    
    return factors

m_fac = factor_matrices(mat)
print(f"{[m.shape for m in m_fac]}")
m_fac

[(2, 2), (2, 2), (2, 2)]


[array([[-0.64142303, -0.7671874 ],
        [-0.7671874 ,  0.64142303]]),
 array([[-0.56672424, -0.82390754],
        [-0.82390754,  0.56672424]]),
 array([[-0.37616823, -0.92655138],
        [-0.92655138,  0.37616823]])]

In [ ]:
def mode_n_product(T, U, mode):
    A = unfold(T, mode)
    result = U.T @ A
    
    # refold mat
    new_shape = list(T.shape)
    new_shape[mode] = U.shape[1]
    return result.reshape(
        [new_shape[mode]] + [new_shape[i] for i in range(len(new_shape)) if i != mode]
    ).swapaxes(0, mode)

def core_tensor(M, factors):
    G = M.copy()
    for mode, U in enumerate(factors):
        G = mode_n_product(G, U, mode)
    return G

G = core_tensor(mat, m_fac)
G

array([[[-1.42253953e+01,  4.61793060e-03],
        [ 1.60125603e-02,  5.43770692e-01]],

       [[ 8.28025332e-03,  1.11585148e+00],
        [ 2.38589095e-01,  2.00114739e-01]]])

In [ ]:
def hosvd(M):
    factors = factor_matrices(M)
    G = core_tensor(M, factors)
    return G, factors

